<a href="https://colab.research.google.com/github/alpacaYiChun/ML/blob/master/ClipLight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Tiny-from-scratch CLIP on parquet-only COCO Karpathy (NO dataset scripts)
# Dataset: yerevann/coco-karpathy (captions in `sentences`, images via `url`)
# Caching: disk + memory (never re-download within the same runtime if disk persists)
#   - Disk cache: /content/img_cache_coco (atomic writes, corruption handling)
#   - RAM cache: small LRU in bytes (avoids repeated disk reads in same run)
# Loss:
#   - i2t: mined multi-positive soft labels (top-m positives, stop-grad, eps over positives only)
#   - t2i: hard CE
# Eval: Recall@50 image->text and text->image
# ============================================================

!pip -q install "datasets>=2.18.0" "pillow" "tqdm" "transformers>=4.38.0" "torchvision" "requests"

import os, io, math, random, time, hashlib
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from collections import OrderedDict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm
import torchvision.transforms as T
from transformers import AutoTokenizer
import requests

# ----------------------------
# Config
# ----------------------------
@dataclass
class CFG:
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    dataset_name: str = "yerevann/coco-karpathy"  # parquet-only
    img_size: int = 224

    images_per_batch: int = 48
    captions_per_image: int = 3
    num_workers: int = 0   # if you see download duplication, set 0

    tokenizer_name: str = "bert-base-uncased"
    max_len: int = 32

    embed_dim: int = 256
    img_width: int = 256
    txt_width: int = 256
    txt_layers: int = 4
    txt_heads: int = 4
    txt_mlp_ratio: int = 4
    dropout: float = 0.1

    epochs: int = 10
    lr: float = 2e-4
    wd: float = 0.05
    warmup_steps: int = 300
    grad_clip: float = 1.0
    amp: bool = True
    logit_scale_max: float = 100.0

    pos_tau: float = 0.07
    pos_top_m: int = 1
    pos_eps: float = 0.05

    max_train_images: Optional[int] = 50000
    eval_max_images: Optional[int] = 5000
    eval_max_captions: Optional[int] = 25000

    # caching
    cache_dir: str = "/content/img_cache_coco"
    ram_cache_bytes: int = 512 * 1024 * 1024  # 512MB RAM cache for image bytes

cfg = CFG()

# ----------------------------
# Repro / device
# ----------------------------
def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_all(cfg.seed)

device = cfg.device
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ----------------------------
# Load dataset
# ----------------------------
ds = load_dataset(cfg.dataset_name)
print("Loaded:", cfg.dataset_name, "splits:", list(ds.keys()))
print("Train columns:", ds["train"].column_names)

# ----------------------------
# Tokenizer
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(cfg.tokenizer_name, use_fast=True)

def encode_text_batch(texts: List[str]) -> torch.Tensor:
    enc = tokenizer(texts, padding="max_length", truncation=True, max_length=cfg.max_len, return_tensors="pt")
    return enc["input_ids"]

# ----------------------------
# Image transforms
# ----------------------------
img_train_tf = T.Compose([
    T.RandomResizedCrop(cfg.img_size, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
img_val_tf = T.Compose([
    T.Resize(cfg.img_size + 32),
    T.CenterCrop(cfg.img_size),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

# ============================================================
# Disk + RAM cache (avoid re-downloads)
# ============================================================
os.makedirs(cfg.cache_dir, exist_ok=True)

def _url_to_path(url: str) -> str:
    h = hashlib.sha1(url.encode("utf-8")).hexdigest()
    return os.path.join(cfg.cache_dir, h + ".jpg")

# RAM LRU in BYTES
class RAMBytesLRU:
    def __init__(self, max_bytes: int):
        self.max_bytes = int(max_bytes)
        self.cur_bytes = 0
        self.od = OrderedDict()  # url -> bytes

    def get(self, url: str) -> Optional[bytes]:
        b = self.od.get(url, None)
        if b is None:
            return None
        self.od.move_to_end(url)
        return b

    def put(self, url: str, b: bytes):
        if b is None:
            return
        size = len(b)
        # if too big, skip caching
        if size > self.max_bytes:
            return
        # replace existing
        if url in self.od:
            old = self.od.pop(url)
            self.cur_bytes -= len(old)
        self.od[url] = b
        self.cur_bytes += size
        self.od.move_to_end(url)
        # evict
        while self.cur_bytes > self.max_bytes and len(self.od) > 0:
            k, oldb = self.od.popitem(last=False)
            self.cur_bytes -= len(oldb)

ram_cache = RAMBytesLRU(cfg.ram_cache_bytes)

def load_image_pil_cached(url: str) -> Image.Image:
    # 1) RAM hit
    b = ram_cache.get(url)
    if b is not None:
        try:
            return Image.open(io.BytesIO(b)).convert("RGB")
        except Exception:
            pass  # fall through

    path = _url_to_path(url)

    # 2) Disk hit
    if os.path.exists(path) and os.path.getsize(path) > 0:
        try:
            with open(path, "rb") as f:
                b = f.read()
            ram_cache.put(url, b)
            return Image.open(io.BytesIO(b)).convert("RGB")
        except Exception:
            # corrupted -> delete so we can re-download once
            try: os.remove(path)
            except: pass

    # 3) Download ONCE, then write atomically to disk, then cache in RAM
    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        b = r.content
        if not b:
            raise RuntimeError("empty content")

        tmp = path + ".tmp"
        with open(tmp, "wb") as f:
            f.write(b)
        os.replace(tmp, path)  # atomic
        ram_cache.put(url, b)
        return Image.open(io.BytesIO(b)).convert("RGB")
    except Exception:
        # fallback image keeps training running
        return Image.new("RGB", (cfg.img_size, cfg.img_size), (0, 0, 0))

# ----------------------------
# Caption extractor
# ----------------------------
def extract_captions(ex: Dict[str, Any]) -> List[str]:
    s = ex.get("sentences", None)
    if isinstance(s, list) and len(s) > 0:
        out = [t.strip() for t in s if isinstance(t, str) and t.strip()]
        if out:
            return out
    raise KeyError(f"No captions found. Keys={list(ex.keys())}")

# ----------------------------
# Dataset
# ----------------------------
class CocoKarpathyURL(Dataset):
    def __init__(self, hf_split, img_tf, max_images: Optional[int] = None):
        self.data = hf_split
        self.img_tf = img_tf
        self.N = len(hf_split) if max_images is None else min(len(hf_split), max_images)
    def __len__(self): return self.N
    def __getitem__(self, idx):
        ex = self.data[idx]
        url = ex["url"]
        caps = extract_captions(ex)
        img = load_image_pil_cached(url)
        return img, caps

train_set = CocoKarpathyURL(ds["train"], img_train_tf, max_images=cfg.max_train_images)

def collate_grouped(batch):
    K = cfg.captions_per_image
    imgs, cap_texts, cap_to_img = [], [], []
    for i, (img, caps) in enumerate(batch):
        imgs.append(train_set.img_tf(img))
        chosen = random.sample(caps, K) if len(caps) >= K else [random.choice(caps) for _ in range(K)]
        for c in chosen:
            cap_texts.append(c)
            cap_to_img.append(i)
    cap_ids = encode_text_batch(cap_texts)  # [N*K, L]
    return torch.stack(imgs, 0), cap_ids, torch.tensor(cap_to_img, dtype=torch.long)

train_loader = DataLoader(
    train_set,
    batch_size=cfg.images_per_batch,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=True,
    collate_fn=collate_grouped,
)

# ----------------------------
# Model
# ----------------------------
class ResidualBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(c)
    def forward(self, x):
        h = F.relu(self.bn1(self.conv1(x)))
        h = self.bn2(self.conv2(h))
        return F.relu(x + h)

class SmallCNN(nn.Module):
    def __init__(self, out_width=256):
        super().__init__()
        c1, c2, c3, c4 = 64, 128, 192, 256
        self.stem = nn.Sequential(
            nn.Conv2d(3, c1, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c1),
            nn.ReLU(inplace=True),
        )
        def stage(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                ResidualBlock(cout),
            )
        self.s1 = stage(c1, c2)
        self.s2 = stage(c2, c3)
        self.s3 = stage(c3, c4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(c4, out_width)
    def forward(self, x):
        x = self.stem(x)
        x = self.s1(x); x = self.s2(x); x = self.s3(x)
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return self.fc(x)

class TextTransformer(nn.Module):
    def __init__(self, vocab_size, width=256, layers=4, heads=4, mlp_ratio=4, max_len=32, dropout=0.1, pad_id=0):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, width)
        self.pos = nn.Embedding(max_len, width)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads,
            dim_feedforward=width * mlp_ratio,
            dropout=dropout, activation="gelu",
            batch_first=True, norm_first=True
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.ln = nn.LayerNorm(width)
        self.pad_id = pad_id
    def forward(self, input_ids):
        B, L = input_ids.shape
        pos_ids = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        x = self.tok(input_ids) + self.pos(pos_ids)
        x = self.enc(x)
        x = self.ln(x)
        not_pad = (input_ids != self.pad_id).int()
        last_idx = (not_pad.sum(dim=1) - 1).clamp_min(0)
        return x[torch.arange(B, device=input_ids.device), last_idx]

class TinyCLIP(nn.Module):
    def __init__(self, vocab_size, pad_id):
        super().__init__()
        self.image = SmallCNN(out_width=cfg.img_width)
        self.text  = TextTransformer(
            vocab_size=vocab_size, width=cfg.txt_width,
            layers=cfg.txt_layers, heads=cfg.txt_heads,
            mlp_ratio=cfg.txt_mlp_ratio, max_len=cfg.max_len,
            dropout=cfg.dropout, pad_id=pad_id
        )
        self.img_proj = nn.Linear(cfg.img_width, cfg.embed_dim, bias=False)
        self.txt_proj = nn.Linear(cfg.txt_width, cfg.embed_dim, bias=False)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1/0.07), dtype=torch.float32))
    def encode_image(self, x): return self.img_proj(self.image(x))
    def encode_text(self, t):  return self.txt_proj(self.text(t))

model = TinyCLIP(vocab_size=tokenizer.vocab_size, pad_id=tokenizer.pad_token_id).to(device)

# ----------------------------
# Loss
# ----------------------------
def mined_multi_positive_i2t_loss(logits_i2t, cap_to_img, tau_pos=0.07, top_m=1, eps_pos=0.0):
    Nimg, Ncap = logits_i2t.shape
    dev = logits_i2t.device

    pos = torch.zeros((Nimg, Ncap), device=dev, dtype=torch.bool)
    pos[cap_to_img, torch.arange(Ncap, device=dev)] = True

    with torch.no_grad():
        pos_logits = logits_i2t.detach().masked_fill(~pos, float("-inf"))
        P = pos.sum(dim=1)
        targets = torch.zeros((Nimg, Ncap), device=dev, dtype=logits_i2t.dtype)

        for i in range(Nimg):
            p = int(P[i].item())
            if p == 0:
                continue
            m = min(top_m, p)
            top_idx = torch.topk(pos_logits[i], k=m, dim=0).indices
            w = F.softmax(pos_logits[i, top_idx] / max(tau_pos, 1e-6), dim=0)
            targets[i, top_idx] = w
            if eps_pos > 0:
                uni = pos[i].to(logits_i2t.dtype)
                uni = uni / uni.sum().clamp_min(1.0)
                targets[i] = (1 - eps_pos) * targets[i] + eps_pos * uni

        targets = targets / targets.sum(dim=1, keepdim=True).clamp_min(1.0)

    logp = F.log_softmax(logits_i2t, dim=1)
    return -(targets * logp).sum(dim=1).mean()

def clip_loss(img_f, txt_f, cap_to_img, logit_scale):
    img_f = F.normalize(img_f, dim=-1)
    txt_f = F.normalize(txt_f, dim=-1)
    scale = logit_scale.exp().clamp(1e-3, cfg.logit_scale_max)
    logits_i2t = scale * (img_f @ txt_f.t())
    loss_i2t = mined_multi_positive_i2t_loss(
        logits_i2t, cap_to_img, tau_pos=cfg.pos_tau, top_m=cfg.pos_top_m, eps_pos=cfg.pos_eps
    )
    loss_t2i = F.cross_entropy(logits_i2t.t(), cap_to_img)
    return 0.5 * (loss_i2t + loss_t2i)

# ----------------------------
# Optim + schedule
# ----------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.98), eps=1e-6, weight_decay=cfg.wd)
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.amp and device=="cuda"))
total_steps = cfg.epochs * len(train_loader)

def lr_for_step(step):
    if step < cfg.warmup_steps:
        return cfg.lr * (step + 1) / max(1, cfg.warmup_steps)
    t = (step - cfg.warmup_steps) / max(1, total_steps - cfg.warmup_steps)
    return cfg.lr * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * min(1.0, t))))

# ----------------------------
# Recall@50 eval (uses same cache; no re-download)
# ----------------------------
@torch.no_grad()
def encode_val_for_retrieval(hf_split):
    model.eval()

    Nimg = len(hf_split)
    if cfg.eval_max_images is not None:
        Nimg = min(Nimg, cfg.eval_max_images)

    images, captions, cap_to_img, img_to_caps = [], [], [], []

    for i in range(Nimg):
        ex = hf_split[i]
        url = ex["url"]
        img = load_image_pil_cached(url)
        caps = extract_captions(ex)

        images.append(img)
        cap_ids = []
        for c in caps:
            if cfg.eval_max_captions is not None and len(captions) >= cfg.eval_max_captions:
                break
            cap_ids.append(len(captions))
            captions.append(c)
            cap_to_img.append(i)
        img_to_caps.append(cap_ids)
        if cfg.eval_max_captions is not None and len(captions) >= cfg.eval_max_captions:
            break

    img_feats = []
    for s in tqdm(range(0, len(images), 128), desc="Encode val images"):
        batch = torch.stack([img_val_tf(images[j]) for j in range(s, min(s+128, len(images)))], 0).to(device)
        with torch.cuda.amp.autocast(enabled=(cfg.amp and device=="cuda")):
            f = model.encode_image(batch)
        img_feats.append(F.normalize(f.float(), dim=-1).cpu())
    img_feats = torch.cat(img_feats, 0)

    cap_feats = []
    for s in tqdm(range(0, len(captions), 256), desc="Encode val captions"):
        toks = tokenizer(
            captions[s:s+256],
            padding="max_length",
            truncation=True,
            max_length=cfg.max_len,
            return_tensors="pt",
        )["input_ids"].to(device)
        with torch.cuda.amp.autocast(enabled=(cfg.amp and device=="cuda")):
            f = model.encode_text(toks)
        cap_feats.append(F.normalize(f.float(), dim=-1).cpu())
    cap_feats = torch.cat(cap_feats, 0)

    logit_scale = model.logit_scale.exp().float().cpu().clamp(1e-3, cfg.logit_scale_max)
    return img_feats, cap_feats, cap_to_img, img_to_caps, logit_scale

@torch.no_grad()
def recall_at_k(img_feats, cap_feats, cap_to_img, img_to_caps, logit_scale, k=50):
    sims = (img_feats @ cap_feats.t()) * logit_scale
    Nimg, Ncap = sims.shape

    topk_caps = torch.topk(sims, k=min(k, Ncap), dim=1).indices
    correct_i2t = 0
    for i in range(Nimg):
        gt = set(img_to_caps[i])
        if any(int(c) in gt for c in topk_caps[i].tolist()):
            correct_i2t += 1
    r_i2t = correct_i2t / max(1, Nimg)

    sims_t = sims.t()
    topk_imgs = torch.topk(sims_t, k=min(k, Nimg), dim=1).indices
    correct_t2i = 0
    for c in range(Ncap):
        if cap_to_img[c] in topk_imgs[c].tolist():
            correct_t2i += 1
    r_t2i = correct_t2i / max(1, Ncap)
    return r_i2t, r_t2i

# ----------------------------
# Train
# ----------------------------
global_step = 0
model.train()

for epoch in range(1, cfg.epochs + 1):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.epochs}")
    ema = None

    for images, input_ids, cap_to_img in pbar:
        images = images.to(device, non_blocking=True)
        input_ids = input_ids.to(device, non_blocking=True)
        cap_to_img = cap_to_img.to(device, non_blocking=True)

        lr = lr_for_step(global_step)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(cfg.amp and device=="cuda")):
            img_f = model.encode_image(images)
            txt_f = model.encode_text(input_ids)
            loss = clip_loss(img_f, txt_f, cap_to_img, model.logit_scale)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        if cfg.grad_clip and cfg.grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

        scaler.step(optimizer)
        scaler.update()

        l = float(loss.item())
        ema = l if ema is None else (0.95 * ema + 0.05 * l)
        pbar.set_postfix(loss=f"{ema:.4f}", lr=f"{lr:.2e}", scale=f"{float(model.logit_scale.exp().item()):.2f}")
        global_step += 1

    img_feats, cap_feats, cap_to_img_val, img_to_caps_val, logit_scale = encode_val_for_retrieval(ds["validation"])
    r_i2t_50, r_t2i_50 = recall_at_k(img_feats, cap_feats, cap_to_img_val, img_to_caps_val, logit_scale, k=50)
    print(f"\n[VAL] Recall@50 image->text: {r_i2t_50:.4f}")
    print(f"[VAL] Recall@50 text->image: {r_t2i_50:.4f}\n")

print("Done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.6/221.6 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.3/377.3 kB 38.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in

README.md:   0%|          | 0.00/246 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/17.0M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

...


[VAL] Recall@50 image->text: 0.4066
[VAL] Recall@50 text->image: 0.3959



Epoch 10/10:   0%|          | 0/1041 [00:00<?, ?it/s]

Encode val images:   0%|          | 0/40 [00:00<?, ?it/s]

Encode val captions:   0%|          | 0/98 [00:00<?, ?it/s]


[VAL] Recall@50 image->text: 0.4066
[VAL] Recall@50 text->image: 0.4050

Done.
